# Offline skill of the buoyancy-flux ANN — figures + playground

Two things in one notebook, run with the **`Pavel_Container`** kernel on a node with `/scratch`:

1. **Figures** for paper §3.3 — R²F/corr heatmaps (factor × depth) from the precomputed skill files.
2. **Playground** — fast *single-slice* predictions + the along/across-gradient decomposition, with region zooms, so you (or I) can poke at spatial patterns interactively (e.g. is along/across error uniform or boundary-concentrated?).

The single-slice path is fast (seconds) — no need to run the full `predict_ANN_rho` loop to explore maps. Edit the **Config** cell and re-run the playground cells.

In [ ]:
import sys
sys.path.append('../src/training-on-CM2.6')
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
try:
    import cmocean
    CMAP = cmocean.cm.balance
except Exception:
    CMAP = 'RdBu_r'

from helpers.cm26 import read_datasets
from helpers.ann_tools import import_ANN
from helpers.selectors import *   # select_globe, select_NA, select_SO, select_Gulf, select_ACC, compare, ...

%load_ext autoreload
%autoreload 2

## Config — edit me

In [ ]:
FACTOR  = 4          # coarse-graining factor to explore (4, 9, 12, 15)
ZL      = 0          # depth-level index (0=surface ... up to 49)
TIME    = 0          # test-snapshot index (0..23)

ANN_PATH  = '../CM26_ML_models/ocean3d/subfilter/FGR3/buoyancy/hidden-layer-32-32/seed-default/model/ann_instance_20Dec.nc'
SKILL_DIR = '/scratch/db194/CM26_ML_models/FGR3/EXP0/skill-test-rho'   # precomputed factor-*.nc (R2F, along/across, maps)
FACTORS   = [4, 9, 12, 15]

In [ ]:
# Load the ANN once; load the test split for the chosen FACTOR (read_datasets reads /scratch/$USER by default).
ann = import_ANN(ANN_PATH)
ds  = read_datasets(['test'], [FACTOR])[f'test-{FACTOR}']
print('loaded factor', FACTOR, '| dims', dict(ds.data.sizes))

## Playground: single-slice prediction + along/across decomposition

Fast (one time, one depth). The along/across split is relative to the local horizontal density gradient $\nabla_h\rho$ (`rhox`,`rhoy`):
- **along** $= \mathbf{F}\cdot\hat n$  (parallel to $\nabla_h\rho$; a downgradient-diffusive flux lives here)
- **across** $= \mathbf{F}\cdot\hat t$  (perpendicular to $\nabla_h\rho$)

*(physical interpretation deliberately left open — this is just the geometry.)*

In [ ]:
def decompose(fx, fy, rhox, rhoy):
    """Return (along, across) components of the horizontal flux relative to grad(rho)."""
    g = np.sqrt(rhox**2 + rhoy**2) + 1e-30
    nx, ny = rhox / g, rhoy / g
    along  = fx * nx + fy * ny
    across = fy * nx - fx * ny
    return along, across

dds  = ds.select2d(zl=ZL, time=TIME)
pred = dds.state.ANN_rho_inference(ann, return_xarray=True)

Fx,  Fy    = dds.data.Fx,       dds.data.Fy
Fxp, Fyp   = pred['Fx_xarray'], pred['Fy_xarray']
rhox, rhoy = dds.data.rhox,     dds.data.rhoy
gmag = np.sqrt(rhox**2 + rhoy**2)

Fa,  Fr  = decompose(Fx,  Fy,  rhox, rhoy)    # truth: along, across
Fap, Frp = decompose(Fxp, Fyp, rhox, rhoy)    # ANN:   along, across
print(f'factor {FACTOR}, zl={ZL}, time={TIME} ready')

### Maps: ANN vs truth vs error — ALONG component (uniform or boundary-concentrated?)
`compare()` prints R²/corr for the chosen region. Swap `sel` (e.g. `select_globe`, `select_Gulf`, `select_ACC`, `select_NA`, `select_Kuroshio`) to localize.

In [ ]:
sel = select_globe   # try select_Gulf, select_ACC, select_NA, select_Kuroshio for boundary regions
plt.figure(figsize=(8,7))
compare(dds.nanvar(Fap,0), dds.nanvar(Fa,0), selector=sel,
        label_test='ANN  (along-gradient)', label_control='CM2.6 truth (along-gradient)')

### Maps: ACROSS component

In [ ]:
plt.figure(figsize=(8,7))
compare(dds.nanvar(Frp,0), dds.nanvar(Fr,0), selector=sel,
        label_test='ANN  (across-gradient)', label_control='CM2.6 truth (across-gradient)')

### Error maps + gradient magnitude (spot weak-gradient artifacts)
Where $|\nabla_h\rho|$ is tiny the along/across split is ill-defined; compare where the error lives against where the gradient is weak.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 3.5))
dds.nanvar(Fa - Fap, 0).plot(ax=ax[0], robust=True, cmap=CMAP); ax[0].set_title('along error (truth-ANN)')
dds.nanvar(Fr - Frp, 0).plot(ax=ax[1], robust=True, cmap=CMAP); ax[1].set_title('across error (truth-ANN)')
np.log10(dds.nanvar(gmag, 0)).plot(ax=ax[2], robust=True);       ax[2].set_title('log10 |grad rho|')
plt.tight_layout()

## Figure 3.3a — skill heatmaps (factor × depth) from precomputed skill files

In [ ]:
sk = {f: xr.open_dataset(f'{SKILL_DIR}/factor-{f}.nc') for f in FACTORS}
spacing = {4:'0.4', 9:'0.9', 12:'1.2', 15:'1.5'}
zl = sk[FACTORS[0]].zl.values

def heat(ax, metric, vmin, vmax, title):
    M = np.stack([sk[f][metric].values for f in FACTORS])   # (nfac, nzl)
    im = ax.imshow(M.T, aspect='auto', cmap=CMAP, vmin=vmin, vmax=vmax, origin='upper')
    ax.set_xticks(range(len(FACTORS))); ax.set_xticklabels([spacing[f] for f in FACTORS])
    yt = np.linspace(0, len(zl)-1, 7).astype(int)
    ax.set_yticks(yt); ax.set_yticklabels([str(int(zl[i])) for i in yt])
    ax.set_xlabel('coarse-grid spacing [deg]'); ax.set_ylabel('depth [m]')
    ax.set_title('%s (mean=%.2f)' % (title, float(np.nanmean(M))))
    return im

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, m, t in zip(ax, ['R2F','R2F_along','R2F_across'], ['$R^2$ combined','$R^2$ along','$R^2$ across']):
    im = heat(a, m, -1, 1, t)
plt.colorbar(im, ax=list(ax)); plt.suptitle('Offline $R^2$ of the buoyancy flux', y=1.03)

### Depth profiles of R² (combined / along / across), one panel per factor

In [ ]:
fig, ax = plt.subplots(1, len(FACTORS), figsize=(15, 4), sharey=True)
for a, f in zip(ax, FACTORS):
    for m, lbl in [('R2F','combined'), ('R2F_along','along'), ('R2F_across','across')]:
        a.plot(sk[f][m], sk[f].zl, label=lbl)
    a.invert_yaxis(); a.set_xlim(-1, 1); a.axvline(0, color='k', lw=0.5)
    a.set_title('$\\Delta$=%s deg' % spacing[f]); a.set_xlabel('$R^2$')
ax[0].set_ylabel('depth [m]'); ax[0].legend()
plt.tight_layout()

## Scratch / playground
Your space — anything goes. (I'll sometimes read this notebook to see what you're investigating.)